# SAÉ — Migration vers NoSQL

## Contexte et contraintes métier

La base décrit la programmation d’une radio et ses diffusions. Les requêtes s’appuient notamment sur les règles suivantes :

- **Protection du jeune public** : aucun titre `explicit` ne doit être diffusé pendant les tranches protégées du matin et de l’après-midi.
- **Quota de nouveautés** : l’antenne doit diffuser un minimum de titres récent·e·s, sélectionné·e·s selon la date de sortie de leur album.
- **Déclaration des diffusions** : pour les sociétés de gestion de droits, il faut pouvoir obtenir rapidement, pour chaque artiste, le nombre de diffusions et la durée cumulée correspondante.

Pour chaque requête, choisissez un ordre de présentation lorsqu’il n’est pas évident, afin de pouvoir comparer les résultats après migration.

In [1]:
!pip install pandas --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Connection to Radio Kiosque database
sqlite_connection = sqlite3.connect("data/RadioKiosque.sqlite")

Comment requêter la base de données ?

In [3]:
pd.read_sql_query(
    """
        SELECT host, title
        FROM Shows
        ORDER BY host
    """,
    con=sqlite_connection
)

,host,title
0,Anouk Lindqvist,Le Réveil Kiosque
1,Anouk Lindqvist,Rush Hour
2,DJ Marvelle,Les Insomniaques
3,Dario Fenn,Tour de Piste
4,Françoise Dubreuil,Dimanche Vinyle
5,Karim Benali,Café Croissant
6,Karim Benali,Nouveautés Kiosque
7,Orla Quinn,Pause Midi
8,Orla Quinn,Weekend Matin
9,Salomé Ndiaye,Scène Ouverte


In [9]:
#Première analyse pour découvrir la base de données

#Nombre de lignes dans quelques tables 

pd.read_sql_query(
    """
        SELECT 
        (SELECT COUNT(*) FROM Shows) AS "Nombre de Shows",
        (SELECT COUNT(*) FROM Broadcasts) AS "Nombre de broadcasts",
        (SELECT COUNT(*) FROM listeners) AS "Nombre d'auditeurs",
        (SELECT COUNT(*) FROM requests) AS "Nombre de requêtes",
        (SELECT COUNT(*) FROM Tracks) AS "Nombre de chansons"
    """,
    con=sqlite_connection
)

,Nombre de Shows,Nombre de broadcasts,Nombre d'auditeurs,Nombre de requêtes,Nombre de chansons
0,12,200107,600,1626,11335


In [13]:
#Afficher quelques lignes de chansons explicites

pd.read_sql_query(
    """
        SELECT name as "Titre", explicit
        FROM Tracks
        WHERE explicit = 1
        LIMIT 10
    """,
    con=sqlite_connection
)

,Titre,explicit
0,Mon amour,1
1,F4ck 4ever,1
2,Girlfriend (feat. Dâm-Funk),1
3,Full of life,1
4,Big eye,1
5,Je pardonne,1
6,Mon cœur tu es fou,1
7,L'antidote,1
8,Il faudrait,1
9,Bonjour docteur,1


### 1. Auditeur·ices sans demande

Lister les auditeur·ices n’ayant jamais effectué de demande musicale.

In [39]:
Q1 = pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

Q1.to_csv("references/q1.csv", index = False)

Q1

,Pseudonyme
0,solar_raven63
1,red_seal79
2,lunar_llama20
3,pink_lynx87
4,red_owl90
...,...
174,big_owl10
175,secret_sparrow56
176,fast_sparrow44
177,calm_raven24


### 2. Bilan par animateur·ice

Pour chaque animateur·ice, indiquer le nombre d’émissions, le nombre de diffusions et la durée totale diffusée.

In [40]:
Q2 = pd.read_sql_query(
    """
        SELECT Shows.host as "Animateur(trice)", 
            COUNT( DISTINCT Shows.show_id) as "Nombre d'émissions", 
            COUNT(Broadcasts.broadcast_id) as "Nombre de diffusions",
            SUM(Tracks.duration_ms) as "Durée totale diffusée"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.host
        ORDER BY Shows.host
    """,
    con=sqlite_connection
)

Q2.to_csv("references/q2.csv", index = False)

Q2

,Animateur(trice),Nombre d'émissions,Nombre de diffusions,Durée totale diffusée
0,Anouk Lindqvist,2,43894,10140191003
1,DJ Marvelle,1,33201,7538676031
2,Dario Fenn,1,21830,5080002640
3,Françoise Dubreuil,1,6466,1510922539
4,Karim Benali,2,28522,6629138076
5,Orla Quinn,2,34592,8107701472
6,Salomé Ndiaye,2,17788,4045055699
7,Yasmine Tazi,1,13814,3016880054


### 3. Bilan par tranche horaire

Pour chaque tranche horaire, indiquer le nombre de diffusions et le nombre de titres explicites diffusés.

In [41]:
Q3 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q3.to_csv("references/q3.csv", index = False)

Q3

,Tranche horaire,Nombre de diffusions,Nombre de titres explicites
0,afternoon,49981,502
1,evening,51766,20210
2,morning,56270,566
3,night,42090,11729


### 4. Bilan par titre

Pour chaque titre, indiquer le nombre de diffusions, l’audience cumulée et le nombre d’émissions différentes où il a été programmé.

In [ ]:
Q4 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q4.to_csv("references/q4.csv", index = False)

Q4

### 5. Bilan par artiste

Pour chaque artiste (principal·e ou invité·e), indiquer le nombre de titres au catalogue, le nombre de diffusions et la durée cumulée diffusée. Inclure les artistes jamais diffusé·es. Trier par ordre décroissant du nombre de diffusions.

In [ ]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

### 6. Diffusions par décennie et tranche horaire

Présenter le nombre de diffusions sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

Voici le format attendu :

| decade | morning | afternoon | evening | night |
|-------:|--------:|----------:|--------:|------:|
|   1960 |         |           |         |       |
|   1970 |         |           |         |       |
|   1980 |         |           |         |       |
|   1990 |         |           |         |       |
|   2000 |         |           |         |       |
|   2010 |         |           |         |       |
|   2020 |         |           |         |       |

&#128161; Exécuter la requête en SQL avec la décennie, la tranche horaire et le nombre de diffusion. Puis pivoter la table avec la fonction `pivot_table` de Pandas.

In [ ]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

### 7. Audience par décennie et tranche horaire

Présenter l’audience cumulée sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

### 8. Top 10 des audiences moyennes

Parmi les titres diffusés au moins trois fois, lister les dix titres ayant la plus forte audience moyenne par diffusion. Trier par l'audience moyenne.

### 9. Infractions à la protection du jeune public

Lister chaque diffusion d’un titre explicite dans une émission protégée, avec le titre, l’identifiant de l’émission et l’horodatage.

### 10. Albums diffusés en entier

Lister les albums dont tous les titres ont été diffusés au moins une fois.

### 11. Podcasts classés par pays

Pour chaque pays, indiquer le nombre de podcasts classés, le nombre de podcasts explicites et le nombre de podcasts dont la fiche est fiable.

### 12. Podcasts programmables

Lister les podcasts programmables, avec leur nom, leur éditeur, le nombre de pays dans le top 10 desquels ils figurent et leur meilleur rang. Trier du plus présent au moins présent, puis par meilleur rang.

### 13. Durée et activité des podcasts programmables

Pour chaque podcast programmable, indiquer le nombre d’épisodes connus, la durée moyenne de ses épisodes (en minutes) et la date de son dernier épisode. Ne garder que les podcasts dont la durée moyenne ne dépasse pas une heure.

In [ ]:
# Close the connection to the database
sqlite_connection.close()